# LAB5 — LangFuse: Observabilidade Completa do Pipeline CRAG

---

| Campo | Detalhe |
|---|---|
| **Aula** | Aula 8 — Avaliação e Observabilidade em RAG |
| **Lab** | LAB5 — Instrumentação CRAG com LangFuse Traces |
| **Objetivo** | Instrumentar o pipeline CRAG do LAB2 com LangFuse para observabilidade completa (latência, tokens, custo, rotas) |
| **Duração estimada** | 60–75 minutos |
| **Pré-requisitos** | LAB2 concluído; conta LangFuse criada (langfuse.com); chaves OpenAI e LangFuse válidas |

---

## 2. Conceitos LangFuse: Hierarquia de Observabilidade

LangFuse organiza a telemetria em uma hierarquia de quatro níveis:

```
TRACE (raiz — representa uma requisição completa ao pipeline)
│
├── SPAN "retrieve" (etapa de recuperação de documentos)
│   └── input: question | output: num_docs
│
├── SPAN "grade_documents" (avaliação de relevância)
│   └── input: docs | output: score_medio | metadata: {scores}
│
├── SPAN "web_search" [opcional — só se docs forem irrelevantes]
│   └── input: question | output: num_results
│
└── GENERATION "generate" (chamada ao LLM)
    └── input: context | output: resposta | usage: {tokens}
```

| Conceito | Descrição |
|---|---|
| **Trace** | Unidade raiz que representa uma chamada completa ao sistema |
| **Span** | Etapa dentro de um trace (ex: recuperação, avaliação, geração) |
| **Generation** | Span especializado para chamadas LLM, com tracking automático de tokens |
| **Score** | Avaliação numérica atribuída a um trace (ex: score de relevância, nota do usuário) |

## 1. Instalação e Configuração

In [ ]:
# Instalar dependências do lab
# langfuse: SDK de observabilidade para aplicações LLM
# langgraph: framework de grafos para agentes LLM
# langchain-openai: integração LangChain com OpenAI
%pip install langfuse langchain langgraph langchain-openai -q

In [ ]:
import os
from google.colab import userdata

# ---- OpenAI ----
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✓ OPENAI_API_KEY carregada")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-..."  # Substitua
    print("⚠ OPENAI_API_KEY definida manualmente")

# ---- LangFuse ----
# Crie uma conta em https://langfuse.com e obtenha as chaves em Settings > API Keys
try:
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
    print("✓ Chaves LangFuse carregadas")
except Exception:
    os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."   # Substitua
    os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."   # Substitua
    print("⚠ Chaves LangFuse definidas manualmente")

# Host do LangFuse — use o cloud ou self-hosted
# Cloud US: https://cloud.langfuse.com
# Cloud EU: https://eu.cloud.langfuse.com
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"
print(f"✓ LANGFUSE_HOST: {os.environ['LANGFUSE_HOST']}")

## 3. Inicializar LangFuse Handler

O `CallbackHandler` do LangFuse integra automaticamente com LangChain. Ao passá-lo como callback, todas as chamadas LLM são automaticamente tracadas — sem necessidade de instrumentação manual linha a linha.

In [ ]:
from langfuse import Langfuse
from langfuse.callback import CallbackHandler

# Inicializar o cliente LangFuse principal
# Usado para instrumentação manual (trace, span, generation)
langfuse = Langfuse(
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    host=os.environ["LANGFUSE_HOST"]
)

# Verificar conectividade com o servidor LangFuse
try:
    langfuse.auth_check()
    print("✓ Conexão com LangFuse estabelecida com sucesso")
except Exception as e:
    print(f"⚠ Erro ao conectar ao LangFuse: {e}")
    print("  Verifique suas chaves e o host configurado")

# Inicializar o CallbackHandler para integração automática com LangChain
# Cada execução de chain/agent com esse handler será automaticamente tracada
langfuse_handler = CallbackHandler(
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    host=os.environ["LANGFUSE_HOST"]
)

print("\n✓ LangFuse CallbackHandler pronto")
print("  Use 'config={'callbacks': [langfuse_handler]}' nas invocações de chain")

## 4. Reproduzir o GraphState e os 4 Nós do LAB2

Reproduzimos a estrutura do CRAG do LAB2 em versão compacta, com as mesmas 4 etapas principais:
1. **retrieve** — busca documentos relevantes no vector store
2. **grade_documents** — avalia a relevância de cada documento
3. **web_search** — busca na web se documentos locais são insuficientes
4. **generate** — gera a resposta final com o LLM

In [ ]:
from typing import TypedDict, List, Optional
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from pydantic import BaseModel, Field
import random

# ===== ESTADO DO GRAFO CRAG =====
# TypedDict define a estrutura do estado que circula entre os nós
class GraphState(TypedDict):
    question: str              # Pergunta original do usuário
    documents: List[Document]  # Documentos recuperados
    generation: str            # Resposta final gerada
    web_search_needed: bool    # Flag: precisa de busca web?
    relevance_scores: List[float]  # Scores de relevância dos documentos


# ===== BASE DE CONHECIMENTO SIMULADA =====
# Em produção, viria de um vector store (ChromaDB, etc.)
BASE_DOCS = [
    Document(
        page_content="""O art. 186 do Código Civil define ato ilícito como a ação ou omissão 
        voluntária, negligência ou imprudência que viola direito e causa dano a outrem.""",
        metadata={"source": "Codigo Civil", "artigo": "186"}
    ),
    Document(
        page_content="""A prescrição para ação de indenização por danos morais é de 3 anos, 
        conforme art. 206, §3º, V do Código Civil brasileiro.""",
        metadata={"source": "Codigo Civil", "artigo": "206"}
    ),
    Document(
        page_content="""O STJ consagrou na Súmula 227 que a pessoa jurídica pode sofrer dano moral, 
        especialmente quando afetada em sua honra objetiva e reputação comercial.""",
        metadata={"source": "STJ", "sumula": "227"}
    ),
    Document(
        page_content="""Flagrante delito: o CPP art. 302 define quatro modalidades — próprio, 
        impróprio, presumido e ficto (este último controverso na doutrina).""",
        metadata={"source": "CPP", "artigo": "302"}
    ),
    Document(
        page_content="""Improbidade administrativa: a Lei 8.429/1992 (com alterações da Lei 14.230/2021) 
        exige dolo específico para configuração do ato ímprobo, conforme nova redação do art. 1º.""",
        metadata={"source": "Lei 8429/1992", "artigo": "1"}
    ),
]

print(f"✓ Base de conhecimento: {len(BASE_DOCS)} documentos")


# ===== MODELOS =====
# LLM para geração de respostas
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Schema para avaliação de relevância
class GradeDoc(BaseModel):
    score: str = Field(description="'yes' se relevante, 'no' se não relevante")
    razao: str = Field(description="Breve justificativa")

# Avaliador de relevância
avaliador = llm.with_structured_output(GradeDoc)
prompt_grade = ChatPromptTemplate.from_messages([
    ("system", "Você avalia se um documento é relevante para uma pergunta jurídica. Responda 'yes' ou 'no'."),
    ("human", "Pergunta: {question}\n\nDocumento: {document}")
])
chain_grade = prompt_grade | avaliador

# Chain de geração
prompt_generate = ChatPromptTemplate.from_messages([
    ("system", "Você é especialista em direito brasileiro. Responda baseado apenas no contexto fornecido."),
    ("human", "Contexto:\n{context}\n\nPergunta: {question}")
])
chain_generate = prompt_generate | llm

print("✓ Modelos e chains configurados")

## 5. Instrumentação Manual com LangFuse

Adicionamos spans explícitos em cada nó do CRAG. Isso nos permite:
- Ver a latência exata de cada etapa
- Registrar inputs/outputs estruturados
- Comparar rotas (local vs web)
- Calcular custo por etapa

In [ ]:
import time
from typing import Optional

# ===== NÓ 1: RETRIEVE =====
def retrieve(state: GraphState, trace=None) -> GraphState:
    """
    Recupera documentos relevantes da base de conhecimento.
    Instrumentado com span LangFuse para rastrear latência e resultados.
    """
    question = state["question"]
    inicio = time.time()
    
    # Simular busca por similaridade (em produção: vector store.similarity_search)
    # Retornamos os primeiros 3 documentos como simulação
    documents = BASE_DOCS[:3]
    
    latencia_ms = (time.time() - inicio) * 1000
    
    # Criar span LangFuse para esta etapa
    if trace:
        span = trace.span(
            name="retrieve",          # Nome da etapa
            input={"question": question},  # Input rastreado
            output={"num_docs": len(documents)},  # Output rastreado
            metadata={
                "latencia_ms": round(latencia_ms, 2),
                "docs_recuperados": [d.metadata.get("source", "?") for d in documents]
            }
        )
        span.end()  # Finalizar o span
    
    print(f"  [retrieve] {len(documents)} docs recuperados ({latencia_ms:.1f}ms)")
    return {**state, "documents": documents}


# ===== NÓ 2: GRADE DOCUMENTS =====
def grade_documents(state: GraphState, trace=None) -> GraphState:
    """
    Avalia a relevância de cada documento recuperado.
    Registra o score médio e todos os scores individuais como metadata.
    """
    question = state["question"]
    documents = state["documents"]
    inicio = time.time()
    
    # Avaliar cada documento individualmente
    scores = []
    docs_relevantes = []
    
    for doc in documents:
        # Chamar o avaliador LLM
        resultado = chain_grade.invoke({
            "question": question,
            "document": doc.page_content
        })
        
        # Converter yes/no para score numérico
        score_num = 1.0 if resultado.score.lower() == "yes" else 0.0
        scores.append(score_num)
        
        if score_num == 1.0:
            docs_relevantes.append(doc)
    
    # Calcular score médio e determinar se precisa de busca web
    score_medio = sum(scores) / len(scores) if scores else 0.0
    web_needed = score_medio < 0.5  # Threshold para ativar busca web
    latencia_ms = (time.time() - inicio) * 1000
    
    # Span LangFuse com scores detalhados
    if trace:
        span = trace.span(
            name="grade_documents",
            input={"num_docs": len(documents)},
            output={"score_medio": round(score_medio, 3), "web_search_needed": web_needed},
            metadata={
                "scores": scores,           # Scores individuais de cada doc
                "latencia_ms": round(latencia_ms, 2),
                "docs_relevantes": len(docs_relevantes),
                "threshold_usado": 0.5
            }
        )
        span.end()
    
    rota = "WEB" if web_needed else "LOCAL"
    print(f"  [grade] score_medio={score_medio:.2f} | rota={rota} ({latencia_ms:.1f}ms)")
    
    # Usar docs relevantes se houver; caso contrário, manter todos para web search
    docs_para_gerar = docs_relevantes if docs_relevantes else documents
    
    return {**state,
            "documents": docs_para_gerar,
            "web_search_needed": web_needed,
            "relevance_scores": scores}


# ===== NÓ 3: WEB SEARCH (SIMULADO) =====
def web_search(state: GraphState, trace=None) -> GraphState:
    """
    Executa busca web quando documentos locais são insuficientes.
    Neste lab usa resultados simulados; em produção, usaria Tavily (LAB4).
    """
    question = state["question"]
    inicio = time.time()
    
    # Simular resultados de busca web
    resultados_web = [
        Document(
            page_content=f"[Web] Resultado simulado para: {question[:50]}. Artigo jurídico recente.",
            metadata={"source": "web_search", "url": "https://exemplo.jus.br"}
        )
    ]
    
    latencia_ms = (time.time() - inicio) * 1000
    
    # Span LangFuse para web search
    if trace:
        span = trace.span(
            name="web_search",
            input={"question": question},
            output={"num_results": len(resultados_web)},
            metadata={
                "latencia_ms": round(latencia_ms, 2),
                "fonte": "simulado (use Tavily em produção)"
            }
        )
        span.end()
    
    print(f"  [web_search] {len(resultados_web)} resultados ({latencia_ms:.1f}ms)")
    
    # Combinar docs locais com resultados web
    docs_combinados = state["documents"] + resultados_web
    return {**state, "documents": docs_combinados}


# ===== NÓ 4: GENERATE =====
def generate(state: GraphState, trace=None) -> GraphState:
    """
    Gera a resposta final usando o LLM com o contexto disponível.
    Usa Generation (não Span) para rastrear tokens e custo.
    """
    question = state["question"]
    documents = state["documents"]
    inicio = time.time()
    
    # Montar contexto concatenando todos os documentos
    context = "\n\n".join([doc.page_content for doc in documents])
    
    # Chamar o LLM para gerar resposta
    resposta = chain_generate.invoke({
        "context": context,
        "question": question
    })
    
    latencia_ms = (time.time() - inicio) * 1000
    
    # Estimar tokens (aproximação sem contador exato)
    prompt_tokens = len(context) // 3 + len(question) // 3
    completion_tokens = len(resposta.content) // 3
    
    # Generation LangFuse — tipo especial de span para chamadas LLM
    # Inclui tracking de tokens e modelo usado
    if trace:
        generation = trace.generation(
            name="generate",
            model="gpt-4o-mini",
            input=context[:500],        # Input truncado para o trace
            output=resposta.content,    # Output completo
            usage={
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens
            },
            metadata={"latencia_ms": round(latencia_ms, 2)}
        )
        generation.end()
    
    print(f"  [generate] {len(resposta.content)} chars | ~{prompt_tokens+completion_tokens} tokens ({latencia_ms:.1f}ms)")
    return {**state, "generation": resposta.content}


print("✓ Todos os 4 nós CRAG definidos com instrumentação LangFuse")

## 6. Executar 5 Queries com Traces LangFuse

Para cada query, criamos um **trace raiz** e passamos para todos os nós. O trace agrupa todos os spans em uma única visualização no dashboard LangFuse.

In [ ]:
# Queries de teste — mistura de temas que ativam diferentes rotas
QUERIES_TESTE = [
    "Qual é o prazo prescricional para ação de indenização por danos morais?",      # Deve encontrar na base local
    "O que é ato ilícito segundo o Código Civil?",                                   # Deve encontrar na base local
    "Quais são as novas súmulas do STJ sobre responsabilidade civil em 2024?",       # Pode ativar web search
    "Como funciona a prisão em flagrante no CPP?",                                   # Deve encontrar na base local
    "Quais são as punições previstas na Lei de IA brasileira de 2024?",              # Deve ativar web search
]

# Armazenar métricas de cada execução
metricas_execucao = []

print("Executando 5 queries com instrumentação LangFuse...")
print("=" * 70)

for i, query in enumerate(QUERIES_TESTE, 1):
    print(f"\nQuery {i}: {query[:65]}..." if len(query) > 65 else f"\nQuery {i}: {query}")
    
    # Registrar tempo total da execução
    inicio_total = time.time()
    
    # ---- CRIAR TRACE RAIZ NO LANGFUSE ----
    # Cada query tem seu próprio trace com ID único
    trace = langfuse.trace(
        name="crag_pipeline",          # Nome do pipeline
        input={"question": query},     # Input do trace
        metadata={
            "query_id": i,
            "aula": "aula8",
            "lab": "LAB5"
        },
        tags=["crag", "aula8", "juridico"]  # Tags para filtragem no dashboard
    )
    
    # Inicializar o estado do grafo
    state: GraphState = {
        "question": query,
        "documents": [],
        "generation": "",
        "web_search_needed": False,
        "relevance_scores": []
    }
    
    # ---- EXECUTAR OS NÓS EM SEQUÊNCIA ----
    state = retrieve(state, trace=trace)
    state = grade_documents(state, trace=trace)
    
    # Nó de web search só é executado se necessário
    if state["web_search_needed"]:
        state = web_search(state, trace=trace)
        rota = "web"
    else:
        rota = "local"
    
    state = generate(state, trace=trace)
    
    # Calcular métricas totais
    latencia_total_ms = (time.time() - inicio_total) * 1000
    tokens_total = sum([
        len(doc.page_content) // 3 for doc in state["documents"]
    ]) + len(state["generation"]) // 3
    
    # ---- ATUALIZAR O TRACE COM OUTPUT E SCORE ----
    trace.update(
        output={"resposta": state["generation"][:200]},  # Output truncado
        metadata={
            "rota": rota,
            "latencia_total_ms": round(latencia_total_ms, 2),
            "tokens_total_estimado": tokens_total,
            "web_search_ativado": state["web_search_needed"]
        }
    )
    
    # Armazenar métricas para análise posterior
    metricas_execucao.append({
        "query_id": i,
        "query": query[:60] + "..." if len(query) > 60 else query,
        "rota": rota,
        "latencia_ms": round(latencia_total_ms, 2),
        "tokens_total": tokens_total,
        "web_search": state["web_search_needed"],
        "trace_id": trace.id
    })
    
    print(f"  → Rota: {rota.upper()} | Latência: {latencia_total_ms:.0f}ms | Tokens: ~{tokens_total}")
    
    # Pequena pausa entre queries para evitar rate limiting
    time.sleep(1)

# Garantir que todos os eventos são enviados ao servidor LangFuse
langfuse.flush()

print("\n" + "=" * 70)
print(f"✓ {len(QUERIES_TESTE)} queries executadas e tracadas no LangFuse")
print(f"  Acesse: {os.environ['LANGFUSE_HOST']} para ver os traces")

## 7. Análise dos Traces

Com os dados coletados, calculamos métricas operacionais do pipeline:

In [ ]:
import pandas as pd

# Criar DataFrame com todas as métricas coletadas
df_metricas = pd.DataFrame(metricas_execucao)

# ===== TAXA DE ATIVAÇÃO DO WEB SEARCH =====
num_web = df_metricas["web_search"].sum()
total_queries = len(df_metricas)
taxa_web = num_web / total_queries

print("=" * 60)
print("ANÁLISE DE TAXA DE ATIVAÇÃO DO WEB SEARCH")
print("=" * 60)
print(f"  Queries com web search: {num_web}/{total_queries}")
print(f"  Taxa de ativação: {taxa_web:.1%}")
print(f"  Queries locais: {total_queries - num_web}/{total_queries} ({1-taxa_web:.1%})")

# ===== LATÊNCIA MÉDIA POR ROTA =====
print("\n" + "=" * 60)
print("LATÊNCIA MÉDIA POR ROTA")
print("=" * 60)

latencia_por_rota = df_metricas.groupby("rota")["latencia_ms"].agg(["mean", "min", "max"])
for rota, row in latencia_por_rota.iterrows():
    print(f"  Rota {rota.upper():5s}: média={row['mean']:.0f}ms | min={row['min']:.0f}ms | max={row['max']:.0f}ms")

# ===== CUSTO ESTIMADO POR ROTA =====
print("\n" + "=" * 60)
print("CUSTO ESTIMADO POR ROTA (gpt-4o-mini)")
print("=" * 60)

# Preços gpt-4o-mini: $0.15/1M input, $0.60/1M output
PRECO_POR_TOKEN = 0.00000015  # Média simplificada: $0.15/1M tokens

df_metricas["custo_usd"] = df_metricas["tokens_total"] * PRECO_POR_TOKEN

custo_por_rota = df_metricas.groupby("rota")["custo_usd"].agg(["mean", "sum"])
for rota, row in custo_por_rota.iterrows():
    print(f"  Rota {rota.upper():5s}: custo médio=${row['mean']:.8f} | total=${row['sum']:.8f}")

print(f"\n  Custo total das 5 queries: ${df_metricas['custo_usd'].sum():.8f} USD")

## 8. Dashboard Textual com Pandas

In [ ]:
# Criar dashboard completo de todas as execuções
print("DASHBOARD — EXECUÇÕES CRAG COM LANGFUSE")
print("=" * 100)

# Selecionar e renomear colunas para exibição
dashboard = df_metricas[[
    "query_id", "query", "rota", "latencia_ms", "tokens_total", "custo_usd"
]].copy()

dashboard.columns = ["ID", "Query", "Rota", "Latência (ms)", "Tokens", "Custo (USD)"]

# Formatar valores numéricos
dashboard["Latência (ms)"] = dashboard["Latência (ms)"].apply(lambda x: f"{x:.0f}")
dashboard["Custo (USD)"] = dashboard["Custo (USD)"].apply(lambda x: f"${x:.8f}")
dashboard["Rota"] = dashboard["Rota"].str.upper()

print(dashboard.to_string(index=False))

# Linha de totais/médias
print("-" * 100)
print(f"{'TOTAL/MÉDIA':40s} {'':6s} {df_metricas['latencia_ms'].mean():.0f} ms avg   "
      f"{df_metricas['tokens_total'].sum():6d} tokens   "
      f"${df_metricas['custo_usd'].sum():.8f} total")

# Adicionar link para o dashboard LangFuse
print("\n" + "=" * 100)
print(f"Dashboard LangFuse: {os.environ['LANGFUSE_HOST']}")
print("Filtre por tag 'aula8' ou 'LAB5' para ver apenas estes traces")

## 9. Análise: Qual Rota é Mais Cara? Qual é Mais Lenta?

In [ ]:
import matplotlib.pyplot as plt

# ===== ANÁLISE COMPARATIVA: LOCAL vs WEB =====

# Separar métricas por rota
df_local = df_metricas[df_metricas["rota"] == "local"]
df_web = df_metricas[df_metricas["rota"] == "web"]

# Calcular médias
media_latencia_local = df_local["latencia_ms"].mean() if len(df_local) > 0 else 0
media_latencia_web = df_web["latencia_ms"].mean() if len(df_web) > 0 else 0
media_custo_local = df_local["custo_usd"].mean() if len(df_local) > 0 else 0
media_custo_web = df_web["custo_usd"].mean() if len(df_web) > 0 else 0
media_tokens_local = df_local["tokens_total"].mean() if len(df_local) > 0 else 0
media_tokens_web = df_web["tokens_total"].mean() if len(df_web) > 0 else 0

print("=" * 60)
print("ANÁLISE COMPARATIVA: LOCAL vs WEB")
print("=" * 60)
print(f"\nLatência média:")
print(f"  LOCAL: {media_latencia_local:.0f} ms")
print(f"  WEB:   {media_latencia_web:.0f} ms")
if media_latencia_local > 0 and media_latencia_web > 0:
    mais_lenta = "WEB" if media_latencia_web > media_latencia_local else "LOCAL"
    diff_lat = abs(media_latencia_web - media_latencia_local)
    print(f"  → Rota {mais_lenta} é mais lenta (+{diff_lat:.0f}ms)")

print(f"\nCusto médio de tokens:")
print(f"  LOCAL: ${media_custo_local:.8f} USD")
print(f"  WEB:   ${media_custo_web:.8f} USD")
if media_custo_local > 0 and media_custo_web > 0:
    mais_cara = "WEB" if media_custo_web > media_custo_local else "LOCAL"
    print(f"  → Rota {mais_cara} é mais cara em tokens")

print(f"\nTokens médios:")
print(f"  LOCAL: {media_tokens_local:.0f} tokens")
print(f"  WEB:   {media_tokens_web:.0f} tokens")

# ===== GRÁFICO COMPARATIVO =====
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

categorias = ["LOCAL", "WEB"]
cores = ["steelblue", "tomato"]

# Gráfico 1: Latência
vals_lat = [media_latencia_local, media_latencia_web]
bars1 = axes[0].bar(categorias, vals_lat, color=cores, edgecolor="black")
for bar, val in zip(bars1, vals_lat):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f"{val:.0f}ms", ha="center", fontsize=11)
axes[0].set_title("Latência Média por Rota", fontsize=12)
axes[0].set_ylabel("Milissegundos (ms)")
axes[0].grid(axis="y", alpha=0.3)

# Gráfico 2: Tokens
vals_tok = [media_tokens_local, media_tokens_web]
bars2 = axes[1].bar(categorias, vals_tok, color=cores, edgecolor="black")
for bar, val in zip(bars2, vals_tok):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{val:.0f}", ha="center", fontsize=11)
axes[1].set_title("Tokens Médios por Rota", fontsize=12)
axes[1].set_ylabel("Tokens (estimado)")
axes[1].grid(axis="y", alpha=0.3)

# Gráfico 3: Distribuição de rotas
rotas_count = df_metricas["rota"].value_counts()
axes[2].pie(
    rotas_count.values,
    labels=[r.upper() for r in rotas_count.index],
    autopct="%1.0f%%",
    colors=cores[:len(rotas_count)],
    startangle=90
)
axes[2].set_title("Distribuição de Rotas", fontsize=12)

plt.suptitle("Dashboard CRAG — Métricas por Rota (LAB5)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("lab5_dashboard_rotas.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Dashboard salvo como lab5_dashboard_rotas.png")

## Conclusão

### O que aprendemos neste lab:

1. **LangFuse** oferece observabilidade de nível de produção para pipelines RAG — sem modificar a lógica de negócio
2. **Traces hierárquicos** (trace → spans → generation) permitem identificar gargalos em cada etapa
3. **Instrumentação manual** é mais trabalhosa que callbacks automáticos, mas dá controle total sobre os metadados capturados
4. **Métricas operacionais** (latência, tokens, custo, taxa web) são essenciais para decisões de otimização em produção

### Perguntas de Reflexão:

1. Olhando o dashboard, em quais tipos de query o web search foi ativado? Faz sentido?

2. Se a rota web é mais cara e lenta, como você **minimizaria** o número de ativações sem comprometer a qualidade?

3. Como você usaria os traces LangFuse para **monitorar degradação** de qualidade ao longo do tempo?

4. Em um sistema com 1.000 queries/dia, qual seria o custo mensal estimado baseado nas métricas deste lab?

---
*MBA em RAG & CAG Aplicados a Direito e Segurança Pública · Aula 8*